<a href="https://colab.research.google.com/github/LeandroLDA/Data-Science-Exercises/blob/main/feijao_safra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dicionário dos dados

| Variável | Tipo de Dado | Descrição de Negócio | Exemplo de Preenchimento |
| :--- | :--- | :--- | :--- |
| **produto** | category | Nome normalizado da variedade e tipo de embalagem do feijão. | feijao comum cores (60 kg) |
| **nivel_comercializacao** | category | Etapa da cadeia produtiva em que a transação ocorreu. | atacado, produtor, varejo |
| **uf** | category | Sigla da Unidade Federativa onde o preço foi coletado. | pb, sc, rj |
| **periodo** | datetime | Mês e ano de referência do fechamento da cotação. | 2023-05-01 |
| **preco_medio** | float | Valor médio de comercialização no período correspondente, em Reais (R$). | 284.11 |  

## Limpeza dos dados

In [1]:
!pip install huggingface_hub pandas openpyxl -q

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
from huggingface_hub import hf_hub_download

In [3]:
caminho_arquivo = hf_hub_download(
    repo_id="LeandroLDA/feijao_safra",
    filename="safra-area-producao-tons.csv",
    repo_type="dataset"
)

safra-area-producao-tons.csv:   0%|          | 0.00/153k [00:00<?, ?B/s]

In [4]:
df = pd.read_csv(caminho_arquivo, header=3, index_col=False)

/tmp/ipykernel_505/1063199142.py:1: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(caminho_arquivo, header=3, index_col=False)


In [5]:
colunas = pd.Series(df.columns)
colunas.mask(df.columns.str.contains("Unnamed"), None, inplace=True)
colunas = colunas.ffill()

In [6]:
safras = df.iloc[0,:].astype(str).values
novas_colunas = colunas + " " + safras
df.columns = novas_colunas

In [7]:
df.drop(df.index[0], axis=0 , inplace=True)

In [8]:
df.head(3)

,Unidade da Federação Unidade da Federação,setembro 2006 1.9 Feijão (1ª Safra),setembro 2006 1.10 Feijão (2ª Safra),setembro 2006 1.11 Feijão (3ª Safra),outubro 2006 1.9 Feijão (1ª Safra),outubro 2006 1.10 Feijão (2ª Safra),outubro 2006 1.11 Feijão (3ª Safra),novembro 2006 1.9 Feijão (1ª Safra),novembro 2006 1.10 Feijão (2ª Safra),novembro 2006 1.11 Feijão (3ª Safra),...,abril 2026 1.9 Feijão (1ª Safra),abril 2026 1.10 Feijão (2ª Safra),abril 2026 1.11 Feijão (3ª Safra),maio 2026 1.9 Feijão (1ª Safra),maio 2026 1.10 Feijão (2ª Safra),maio 2026 1.11 Feijão (3ª Safra),junho 2026 1.9 Feijão (1ª Safra),junho 2026 1.10 Feijão (2ª Safra),junho 2026 1.11 Feijão (3ª Safra),julho 2026 1.9 Feijão (1ª Safra)
1,Rondônia,37193,-,-,36741,-,-,36618,-,-,...,1543,547,-,3646,1501,-,3445,1767,-,3048
2,Acre,-,8267,-,-,6753,-,-,6753,-,...,-,2769,-,-,2769,-,-,2769,-,-
3,Amazonas,-,3719,-,-,3719,-,-,4473,-,...,1044,21,-,1044,21,-,1047,21,-,1193


In [9]:
df = df.rename(columns={df.columns[0]:"uf"})

In [10]:
colunas = pd.Series(df.columns)
colunas = (
    colunas
    .str.replace(r'\d+\.\d+\s',"", regex=True)
    .str.replace("Feijão", "")
    .str.replace(r"\((\d)ª Safra\)", r'\1 safra', regex=True)
    .str.lower()
    .str.replace(r"\s+","_", regex=True)
    )

In [11]:
df.columns = colunas

In [12]:
df["uf"] = (
    df["uf"]
    .str.lower()
    .str.replace("\s+","_", regex=True)
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_505/877318734.py:4: SyntaxWarning: invalid escape sequence '\s'
  .str.replace("\s+","_", regex=True)


In [13]:
producao_tons_feijao = df.iloc[:27,:].copy()
dic_producao_tons_feijao = df.iloc[38:,:2].copy()

In [14]:
producao_tons_feijao.set_index("uf")

,setembro_2006_1_safra,setembro_2006_2_safra,setembro_2006_3_safra,outubro_2006_1_safra,outubro_2006_2_safra,outubro_2006_3_safra,novembro_2006_1_safra,novembro_2006_2_safra,novembro_2006_3_safra,dezembro_2006_1_safra,...,abril_2026_1_safra,abril_2026_2_safra,abril_2026_3_safra,maio_2026_1_safra,maio_2026_2_safra,maio_2026_3_safra,junho_2026_1_safra,junho_2026_2_safra,junho_2026_3_safra,julho_2026_1_safra
uf,,,,,,,,,,,,,,,,,,,,,
rondonia,37193,-,-,36741,-,-,36618,-,-,36621,...,1543,547,-,3646,1501,-,3445,1767,-,3048
acre,-,8267,-,-,6753,-,-,6753,-,-,...,-,2769,-,-,2769,-,-,2769,-,-
amazonas,-,3719,-,-,3719,-,-,4473,-,-,...,1044,21,-,1044,21,-,1047,21,-,1193
roraima,-,658,-,-,658,-,-,658,-,-,...,821,-,-,821,-,-,821,-,-,821
para,-,66984,-,-,63175,-,-,63175,-,-,...,5509,7407,-,5509,7407,-,3886,7902,-,3886
amapa,-,635,-,-,850,-,-,850,-,-,...,-,722,-,-,702,-,-,691,-,-
tocantins,5233,2559,-,5233,2559,-,5233,2559,-,5233,...,3659,78790,3759,3832,83149,3759,3832,83311,3759,3832
maranhao,14776,24784,-,14776,24784,-,14776,24784,-,14776,...,8283,21004,-,8287,21004,-,8327,19054,-,8325
piaui,61242,7109,-,61242,7109,-,61184,5875,-,61184,...,74824,6590,-,63773,6687,-,63773,6687,-,40298


In [15]:
producao_long = producao_tons_feijao.melt(
    id_vars = "uf",
    var_name = "periodo_safra",
    value_name = "quantidade_tons"
)

In [16]:
producao_long

,uf,periodo_safra,quantidade_tons
0,rondonia,setembro_2006_1_safra,37193
1,acre,setembro_2006_1_safra,-
2,amazonas,setembro_2006_1_safra,-
3,roraima,setembro_2006_1_safra,-
4,para,setembro_2006_1_safra,-
...,...,...,...
19300,rio_grande_do_sul,julho_2026_1_safra,42732
19301,mato_grosso_do_sul,julho_2026_1_safra,936
19302,mato_grosso,julho_2026_1_safra,14668
19303,goias,julho_2026_1_safra,80135


In [17]:
producao_long[["mes", "ano", "safra", "descartar"]] = producao_long.periodo_safra.str.split("_", expand =True)

In [18]:
producao_long.drop(columns = ["periodo_safra", "descartar"], inplace=True)

In [19]:
producao_long

,uf,quantidade_tons,mes,ano,safra
0,rondonia,37193,setembro,2006,1
1,acre,-,setembro,2006,1
2,amazonas,-,setembro,2006,1
3,roraima,-,setembro,2006,1
4,para,-,setembro,2006,1
...,...,...,...,...,...
19300,rio_grande_do_sul,42732,julho,2026,1
19301,mato_grosso_do_sul,936,julho,2026,1
19302,mato_grosso,14668,julho,2026,1
19303,goias,80135,julho,2026,1


In [20]:
producao_long["quantidade_tons"] = producao_long["quantidade_tons"].replace("-", "0")
producao_long["quantidade_tons"] = pd.to_numeric(producao_long["quantidade_tons"], errors = 'coerce')
producao_long = producao_long.dropna(subset=["quantidade_tons"])

In [21]:
producao_long["uf"] = producao_long["uf"].astype("category")

In [22]:
producao_long[["ano", "safra"]] = producao_long[["ano", "safra"]].astype(int)

In [23]:
dic_meses = {"janeiro":1,"fevereiro":2, "marco":3, "abril":4,
             "maio":5, "junho":6, "julho":7, "agosto":8,
             "setembro":9, "outubro":10, "novembro":11, "dezembro":12
             }

In [24]:
producao_long[producao_long["mes"].isna()]

,uf,quantidade_tons,mes,ano,safra


In [25]:
producao_long["mes"].unique()

array(['setembro', 'outubro', 'novembro', 'dezembro', 'janeiro',
       'fevereiro', 'março', 'abril', 'maio', 'junho', 'julho', 'agosto'],
      dtype=object)

In [26]:
producao_long["mes"] = (
    producao_long["mes"]
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

In [27]:
producao_long["mes"] = producao_long["mes"].map(dic_meses)

In [28]:
producao_long[producao_long["mes"].isna()]

,uf,quantidade_tons,mes,ano,safra


In [29]:
producao_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19305 entries, 0 to 19304
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   uf               19305 non-null  category
 1   quantidade_tons  19305 non-null  int64   
 2   mes              19305 non-null  int64   
 3   ano              19305 non-null  int64   
 4   safra            19305 non-null  int64   
dtypes: category(1), int64(4)
memory usage: 623.5 KB


In [30]:
producao_long


,uf,quantidade_tons,mes,ano,safra
0,rondonia,37193,9,2006,1
1,acre,0,9,2006,1
2,amazonas,0,9,2006,1
3,roraima,0,9,2006,1
4,para,0,9,2006,1
...,...,...,...,...,...
19300,rio_grande_do_sul,42732,7,2026,1
19301,mato_grosso_do_sul,936,7,2026,1
19302,mato_grosso,14668,7,2026,1
19303,goias,80135,7,2026,1


In [31]:
producao_long.to_csv("producao_feijao_tons.csv", sep=",")

In [32]:
producao_long.groupby("uf")["quantidade_tons"].sum()

/tmp/ipykernel_505/2788912122.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  producao_long.groupby("uf")["quantidade_tons"].sum()


,quantidade_tons
uf,
acre,1061041
alagoas,5831188
amapa,279127
amazonas,1258874
bahia,67813353
ceara,35832220
distrito_federal,9794676
espirito_santo,2986669
goias,74215578




---



## Dicionário dos dados

In [33]:
# dic_producao_tons_feijao.rename(columns = {df.columns[0]:"simbolo", df.columns[1]:"descricao"})

In [34]:
# dic_producao_tons_feijao.to_csv("dic_producao_tons_feijao.csv", sep=",")